# Entropía de von Neumann e índices de Zagreb y Wiener en árboles

Se estudian las clases de isomorfismo de árboles de orden $n$ mediante la entropía de von Neumann $S(T)$, el primer índice de Zagreb $Z_1(T)$ y el índice de Wiener $W(T)$. Se comparan sus capacidades discriminantes y se analizan coincidencias numéricas de entropía entre espectros laplacianos distintos.

Los invariantes se almacenan en SQLite; los resultados se presentan en tablas y figuras.


## 1. Configuración


In [ ]:
import os
import csv
import gc
import math
import sqlite3
from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# Parámetros del estudio

NUM_VERTICES = 23

EPS_ENTROPIA = 1e-10
EPS_ESPECTRO = 1e-10
TOL_CERO = 1e-12

# Intervalo de órdenes
ORDEN_MIN_TENDENCIA = 2
ORDEN_MAX_TENDENCIA = NUM_VERTICES
CALCULAR_TENDENCIA = True

# Registro del avance
PROGRESO_CADA = 100000
TAMANO_LOTE = 10000

# Tamaño de la muestra para visualización; estadísticas exhaustivas.
MAX_PUNTOS_GRAFICO = 50000

# Exportación opcional del registro individual de árboles.
EXPORTAR_CSV_MASIVO = False

# Límite de filas para tablas de Excel.
MAX_FILAS_EXCEL = 1_048_576


# Directorios de resultados

CARPETA_SALIDA = f"resultados_seminario_orden_{NUM_VERTICES}"
CARPETA_DATOS = os.path.join(CARPETA_SALIDA, "datos")
CARPETA_TABLAS = os.path.join(CARPETA_SALIDA, "tablas")
CARPETA_IMAGENES = os.path.join(CARPETA_SALIDA, "imagenes")

for carpeta in (
    CARPETA_SALIDA,
    CARPETA_DATOS,
    CARPETA_TABLAS,
    CARPETA_IMAGENES,
):
    os.makedirs(carpeta, exist_ok=True)

print("NetworkX:", nx.__version__)
print("Orden principal:", NUM_VERTICES)
print("Tolerancia de entropía:", EPS_ENTROPIA)
print("Tolerancia espectral:", EPS_ESPECTRO)
print(
    "Órdenes para la tendencia:",
    f"{ORDEN_MIN_TENDENCIA} a {ORDEN_MAX_TENDENCIA}"
)

## 2. Definiciones y cálculo de invariantes

Para un árbol $T$ de orden $n$, con laplaciano $L(T)$, se define

$$
\rho_T=\frac{L(T)}{2(n-1)},\qquad
S(T)=-\sum_{p_i>0}p_i\log_2 p_i,
$$

donde $p_i$ son los autovalores de $\rho_T$. Asimismo,

$$
Z_1(T)=\sum_{v\in V(T)}d(v)^2,\qquad
W(T)=\sum_{\{u,v\}\subseteq V(T)}d_T(u,v).
$$

En árboles, $W(T)$ se calcula a partir de las contribuciones $s(n-s)$ de sus aristas.


In [ ]:
def laplaciano(G):
    """Calcula la matriz laplaciana L = D - A."""
    n = G.number_of_nodes()
    L = np.zeros((n, n), dtype=float)

    for v, d in G.degree():
        L[v, v] = d

    for u, v in G.edges():
        L[u, v] = -1.0
        L[v, u] = -1.0

    return L


def espectro_laplaciano(G):
    """Autovalores laplacianos en orden creciente."""
    valores = np.linalg.eigvalsh(laplaciano(G))
    valores[np.abs(valores) < TOL_CERO] = 0.0
    return valores


def entropia_desde_espectro(espectro, n):
    """Entropía de von Neumann, en bits."""
    normalizacion = 2 * (n - 1)

    p = np.asarray(
        espectro,
        dtype=float
    ) / normalizacion

    p = p[p > TOL_CERO]

    return float(
        -np.sum(p * np.log2(p))
    )


def codigo_arbol(T):
    """Representa el árbol mediante graph6."""
    return (
        nx.to_graph6_bytes(
            T,
            header=False
        )
        .decode("ascii")
        .strip()
    )


def arbol_desde_codigo(codigo):
    """Reconstruye un árbol desde graph6."""
    return nx.from_graph6_bytes(
        codigo.encode("ascii")
    )


def son_cospectrales(
    espectro1,
    espectro2,
    tol=EPS_ESPECTRO
):
    """Determina coincidencia espectral numérica según la tolerancia."""
    return np.allclose(
        espectro1,
        espectro2,
        atol=tol,
        rtol=0.0
    )


def indice_wiener_arbol(T):
    """Calcula Wiener en O(n) mediante las contribuciones s(n-s)."""
    n = T.number_of_nodes()

    if n <= 1:
        return 0

    raiz = 0
    padre = [-2] * n
    padre[raiz] = -1
    orden = [raiz]

    for u in orden:
        for v in T[u]:
            if v != padre[u]:
                padre[v] = u
                orden.append(v)

    tamanio = [1] * n
    W = 0

    for v in reversed(orden[1:]):
        s = tamanio[v]
        W += s * (n - s)
        tamanio[padre[v]] += s

    return int(W)


def invariantes_basicos(T):
    """Calcula Zagreb y Wiener para un árbol."""
    zagreb1 = int(
        sum(
            d**2
            for _, d in T.degree()
        )
    )

    return {
        "zagreb1": zagreb1,
        "wiener": indice_wiener_arbol(T),
    }

## 3. Almacenamiento y enumeración de árboles

Cada árbol se registra mediante su índice, código `graph6`, entropía, Zagreb y Wiener. La enumeración se almacena por lotes en SQLite, sin conservar todos los espectros en memoria. Los registros permiten reanudar una generación interrumpida.


In [ ]:
def ruta_base_orden(n):
    return os.path.join(
        CARPETA_DATOS,
        f"arboles_orden_{n}.sqlite"
    )


def conectar_base(ruta):
    """Abre SQLite con almacenamiento temporal en disco."""
    con = sqlite3.connect(ruta)
    con.row_factory = sqlite3.Row

    con.execute("PRAGMA synchronous = OFF")
    con.execute("PRAGMA journal_mode = OFF")
    con.execute("PRAGMA temp_store = FILE")
    con.execute("PRAGMA cache_size = -65536")

    return con


def crear_estructura_base(con, n):
    con.execute("""
        CREATE TABLE IF NOT EXISTS arboles (
            indice INTEGER PRIMARY KEY,
            codigo TEXT NOT NULL,
            entropia REAL NOT NULL,
            zagreb1 INTEGER NOT NULL,
            wiener INTEGER NOT NULL
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS metadata (
            clave TEXT PRIMARY KEY,
            valor TEXT NOT NULL
        )
    """)

    con.execute(
        """
        INSERT OR REPLACE INTO metadata(clave, valor)
        VALUES('n', ?)
        """,
        (str(n),)
    )

    con.execute(
        """
        INSERT OR IGNORE INTO metadata(clave, valor)
        VALUES('completa', '0')
        """
    )

    con.commit()


def metadata_base(con, clave, defecto=None):
    fila = con.execute(
        "SELECT valor FROM metadata WHERE clave = ?",
        (clave,)
    ).fetchone()

    return (
        fila["valor"]
        if fila is not None
        else defecto
    )


def crear_indices_base(con):
    print("Creación de índices SQLite...")

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_entropia
        ON arboles(entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_zagreb_entropia
        ON arboles(zagreb1, entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_wiener_entropia
        ON arboles(wiener, entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_zagreb_wiener
        ON arboles(zagreb1, wiener)
    """)

    con.commit()


def generar_base_orden(
    n,
    mostrar_progreso=True
):
    ruta = ruta_base_orden(n)
    con = conectar_base(ruta)
    crear_estructura_base(con, n)

    completa = (
        metadata_base(con, "completa", "0")
        == "1"
    )

    if completa:
        total = con.execute(
            "SELECT COUNT(*) AS c FROM arboles"
        ).fetchone()["c"]

        con.close()

        print(
            f"n={n}: base completa reutilizada "
            f"({total:,} árboles)."
        )

        return ruta

    total_esperado = (
        nx.number_of_nonisomorphic_trees(n)
    )

    total_actual = int(
        con.execute(
            "SELECT COUNT(*) AS c FROM arboles"
        ).fetchone()["c"]
    )

    fila = con.execute(
        "SELECT MAX(indice) AS m FROM arboles"
    ).fetchone()

    ultimo = fila["m"]

    inicio = (
        int(ultimo) + 1
        if ultimo is not None
        else 0
    )

    # Se exige continuidad de los índices de la enumeración.
    if total_actual != inicio:
        con.close()
        raise RuntimeError(
            f"La base parcial de n={n} no tiene índices consecutivos. "
            "La base requiere revisión antes de continuar."
        )

    if total_actual == total_esperado:
        print(
            f"n={n}: todos los datos ya estaban guardados. "
            "Verificación de índices SQLite."
        )

    else:
        if inicio > 0:
            print(
                f"n={n}: reanudando desde el índice "
                f"{inicio:,}."
            )

        lote = []

        for indice, T in enumerate(
            nx.nonisomorphic_trees(n)
        ):
            if indice < inicio:
                continue

            espectro = espectro_laplaciano(T)

            invariantes = invariantes_basicos(T)

            lote.append((
                indice,
                codigo_arbol(T),
                entropia_desde_espectro(
                    espectro,
                    n
                ),
                invariantes["zagreb1"],
                invariantes["wiener"],
            ))

            if len(lote) >= TAMANO_LOTE:
                con.executemany(
                    """
                    INSERT OR REPLACE INTO arboles
                    (indice, codigo, entropia, zagreb1, wiener)
                    VALUES (?, ?, ?, ?, ?)
                    """,
                    lote
                )

                con.commit()
                lote.clear()

            cantidad = indice + 1

            if (
                mostrar_progreso
                and PROGRESO_CADA > 0
                and cantidad % PROGRESO_CADA == 0
            ):
                print(
                    f"n={n}: {cantidad:,} / "
                    f"{total_esperado:,} árboles procesados"
                )

        if lote:
            con.executemany(
                """
                INSERT OR REPLACE INTO arboles
                (indice, codigo, entropia, zagreb1, wiener)
                VALUES (?, ?, ?, ?, ?)
                """,
                lote
            )
            con.commit()

    total = con.execute(
        "SELECT COUNT(*) AS c FROM arboles"
    ).fetchone()["c"]

    if total != total_esperado:
        con.close()
        raise RuntimeError(
            f"La base de n={n} tiene {total:,} filas, "
            f"pero se esperaban {total_esperado:,}."
        )

    crear_indices_base(con)

    con.execute(
        """
        INSERT OR REPLACE INTO metadata(clave, valor)
        VALUES('completa', '1')
        """
    )
    con.commit()
    con.close()

    print(
        f"n={n}: generación terminada. "
        f"{total:,} árboles guardados en disco."
    )

    return ruta

def obtener_muestra_base(
    ruta,
    max_puntos=MAX_PUNTOS_GRAFICO
):
    con = conectar_base(ruta)

    N = con.execute(
        "SELECT COUNT(*) AS c FROM arboles"
    ).fetchone()["c"]

    paso = max(
        1,
        math.ceil(N / max_puntos)
    )

    consulta = """
        SELECT
            indice,
            codigo,
            entropia,
            zagreb1,
            wiener
        FROM arboles
        WHERE indice % ? = 0
        ORDER BY indice
        LIMIT ?
    """

    df = pd.read_sql_query(
        consulta,
        con,
        params=(paso, max_puntos)
    )

    con.close()

    return df

## 4. Agrupación por entropía y espectro laplaciano

Las clases numéricas se forman por orden creciente de entropía, con amplitud

$$
S_{\max}-S_{\min}\leq\varepsilon.
$$

Este criterio define una partición numérica; la proximidad $|S_1-S_2|\leq\varepsilon$ no es transitiva. Para árboles de orden fijo se cumplen

$$
\sum_i\lambda_i^2=Z_1(T)+2(n-1),\qquad
W(T)=n\sum_{i=2}^{n}\frac{1}{\lambda_i}.
$$

En consecuencia, diferencias en $Z_1$ o $W$ descartan la cospectralidad. La comparación numérica de espectros se reserva para coincidencias de $(Z_1,W)$.


In [ ]:
def registro_desde_fila(fila):
    return {
        "indice": int(fila["indice"]),
        "codigo": fila["codigo"],
        "entropia": float(fila["entropia"]),
        "zagreb1": int(fila["zagreb1"]),
        "wiener": int(fila["wiener"]),
    }


def iterar_grupos_entropia_cursor(
    cursor,
    tol=EPS_ENTROPIA,
    etiqueta=None
):
    grupo = []
    minimo_grupo = None
    procesados = 0
    siguiente_aviso = PROGRESO_CADA

    for fila in cursor:
        r = registro_desde_fila(fila)
        S = r["entropia"]

        if not grupo:
            grupo = [r]
            minimo_grupo = S

        elif S - minimo_grupo <= tol:
            grupo.append(r)

        else:
            yield grupo
            grupo = [r]
            minimo_grupo = S

        procesados += 1

        if (
            etiqueta is not None
            and PROGRESO_CADA > 0
            and procesados >= siguiente_aviso
        ):
            print(
                f"{etiqueta}: "
                f"{procesados:,} filas analizadas"
            )

            siguiente_aviso += PROGRESO_CADA

    if grupo:
        yield grupo


def obtener_espectro_registro(registro):
    T = arbol_desde_codigo(
        registro["codigo"]
    )

    return espectro_laplaciano(T)


def agrupar_subgrupo_por_espectro(
    registros,
    tol=EPS_ESPECTRO
):
    """Agrupa espectros numéricos con Zagreb y Wiener fijos."""
    clases = []
    representantes = []

    for r in registros:
        espectro = obtener_espectro_registro(r)

        colocado = False

        for k, representante in enumerate(
            representantes
        ):
            if son_cospectrales(
                espectro,
                representante,
                tol=tol
            ):
                clases[k].append(r)
                colocado = True
                break

        if not colocado:
            representantes.append(espectro)
            clases.append([r])

    return clases


def agrupar_por_espectro_optimizado(
    registros,
    tol=EPS_ESPECTRO
):
    """Separa clases espectrales usando Zagreb y Wiener como filtros."""
    por_ZW = defaultdict(list)

    for r in registros:
        por_ZW[
            (r["zagreb1"], r["wiener"])
        ].append(r)

    clases = []

    for subgrupo in por_ZW.values():
        if len(subgrupo) == 1:
            clases.append(subgrupo)
        else:
            clases.extend(
                agrupar_subgrupo_por_espectro(
                    subgrupo,
                    tol=tol
                )
            )

    return clases


def numero_pares_en_clases(clases):
    return sum(
        len(clase) * (len(clase) - 1) // 2
        for clase in clases
    )


def contar_pares_mismo_valor(
    registros,
    campos
):
    if isinstance(campos, str):
        campos = (campos,)

    conteos = Counter(
        tuple(r[c] for c in campos)
        for r in registros
    )

    return sum(
        cantidad * (cantidad - 1) // 2
        for cantidad in conteos.values()
    )


def pares_separados_por_campos(
    clase,
    campos
):
    total = (
        len(clase)
        * (len(clase) - 1)
        // 2
    )

    if total == 0:
        return 0, np.nan

    iguales = contar_pares_mismo_valor(
        clase,
        campos
    )

    separados = total - iguales

    return (
        separados,
        100 * separados / total
    )


def correlacion_pearson_listas(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if (
        len(x) < 2
        or np.std(x) == 0
        or np.std(y) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(x, y)[0, 1]
    )




## 5. Análisis de las clases de entropía

Se obtienen las clases numéricas de entropía, sus multiplicidades y el número de espectros distintos. Se estudian las clases con coincidencias de entropía y, en particular, aquellas con espectros diferentes. La discriminación por $Z_1$, $W$ y $(Z_1,W)$ se calcula respecto al número de espectros.


In [ ]:
COLUMNAS_ENTROPIA_FIJA = [
    "clase_entropia",
    "n_arboles",
    "entropia_representativa",
    "entropia_min",
    "entropia_max",
    "rango_entropia",
    "n_espectros_distintos",
    "pares_totales",
    "pares_cospectrales",
    "pares_no_cospectrales",
    "zagreb_min",
    "zagreb_max",
    "rango_zagreb",
    "n_zagreb_distintos",
    "wiener_min",
    "wiener_max",
    "rango_wiener",
    "n_wiener_distintos",
    "n_valores_Z1_W_distintos",
    "pares_separados_Z1",
    "porcentaje_pares_separados_Z1",
    "pares_separados_W",
    "porcentaje_pares_separados_W",
    "pares_separados_Z1_W",
    "porcentaje_pares_separados_Z1_W",
]


COLUMNAS_CE = [
    "clase_entropia",
    "n_arboles",
    "n_espectros_distintos",
    "entropia_representativa",
    "entropia_min",
    "entropia_max",
    "rango_entropia",
    "zagreb_min",
    "zagreb_max",
    "rango_zagreb",
    "n_zagreb_distintos",
    "wiener_min",
    "wiener_max",
    "rango_wiener",
    "n_wiener_distintos",
    "n_valores_Z1_W_distintos",
    "D_Z1_espectral",
    "D_W_espectral",
    "D_Z1_W_espectral",
    "perdida_espectral_por_entropia",
    "recuperacion_perdida_Z1",
    "recuperacion_perdida_W",
    "recuperacion_perdida_Z1_W",
    "pares_espectrales",
    "pares_espectrales_mismo_Z1",
    "pares_espectrales_mismo_W",
    "pares_espectrales_mismo_Z1_W",
    "pares_espectrales_separados_Z1",
    "pct_pares_espectrales_separados_Z1",
    "pares_espectrales_separados_W",
    "pct_pares_espectrales_separados_W",
    "pares_espectrales_separados_Z1_W",
    "pct_pares_espectrales_separados_Z1_W",
    "correlacion_pearson_Z1_W",
    "max_multiplicidad_cospectral",
]


COLUMNAS_REPRESENTANTES_CE = [
    "clase_entropia",
    "clase_espectral_local",
    "multiplicidad_cospectral",
    "indice_representante",
    "codigo_representante",
    "entropia",
    "zagreb1",
    "wiener",
]


COLUMNAS_COLISIONES_CE = [
    "clase_entropia",
    "entropia_representativa",
    "indice1",
    "indice2",
    "codigo1",
    "codigo2",
    "zagreb1",
    "wiener",
    "delta_entropia",
]


def analizar_clase_ce(
    numero,
    clase,
    clases_espectrales
):
    Q = len(clases_espectrales)

    reps = [
        c[0]
        for c in clases_espectrales
    ]

    zagreb = [
        r["zagreb1"]
        for r in reps
    ]

    wiener = [
        r["wiener"]
        for r in reps
    ]

    n_Z1 = len(set(zagreb))
    n_W = len(set(wiener))
    n_Z1_W = len(
        set(zip(zagreb, wiener))
    )

    pares_espectrales = (
        Q * (Q - 1) // 2
    )

    pares_mismo_Z1 = (
        contar_pares_mismo_valor(
            reps,
            "zagreb1"
        )
    )

    pares_mismo_W = (
        contar_pares_mismo_valor(
            reps,
            "wiener"
        )
    )

    pares_mismo_Z1_W = (
        contar_pares_mismo_valor(
            reps,
            ("zagreb1", "wiener")
        )
    )

    perdida = Q - 1

    entropias = [
        r["entropia"]
        for r in clase
    ]

    fila = {
        "clase_entropia": numero,
        "n_arboles": len(clase),
        "n_espectros_distintos": Q,
        "entropia_representativa":
            float(np.mean(entropias)),
        "entropia_min": min(entropias),
        "entropia_max": max(entropias),
        "rango_entropia":
            max(entropias) - min(entropias),
        "zagreb_min": min(zagreb),
        "zagreb_max": max(zagreb),
        "rango_zagreb":
            max(zagreb) - min(zagreb),
        "n_zagreb_distintos": n_Z1,
        "wiener_min": min(wiener),
        "wiener_max": max(wiener),
        "rango_wiener":
            max(wiener) - min(wiener),
        "n_wiener_distintos": n_W,
        "n_valores_Z1_W_distintos":
            n_Z1_W,
        "D_Z1_espectral":
            n_Z1 / Q,
        "D_W_espectral":
            n_W / Q,
        "D_Z1_W_espectral":
            n_Z1_W / Q,
        "perdida_espectral_por_entropia":
            perdida,
        "recuperacion_perdida_Z1":
            (n_Z1 - 1) / perdida,
        "recuperacion_perdida_W":
            (n_W - 1) / perdida,
        "recuperacion_perdida_Z1_W":
            (n_Z1_W - 1) / perdida,
        "pares_espectrales":
            pares_espectrales,
        "pares_espectrales_mismo_Z1":
            pares_mismo_Z1,
        "pares_espectrales_mismo_W":
            pares_mismo_W,
        "pares_espectrales_mismo_Z1_W":
            pares_mismo_Z1_W,
        "pares_espectrales_separados_Z1":
            pares_espectrales
            - pares_mismo_Z1,
        "pct_pares_espectrales_separados_Z1":
            (
                100
                * (
                    pares_espectrales
                    - pares_mismo_Z1
                )
                / pares_espectrales
                if pares_espectrales
                else np.nan
            ),
        "pares_espectrales_separados_W":
            pares_espectrales
            - pares_mismo_W,
        "pct_pares_espectrales_separados_W":
            (
                100
                * (
                    pares_espectrales
                    - pares_mismo_W
                )
                / pares_espectrales
                if pares_espectrales
                else np.nan
            ),
        "pares_espectrales_separados_Z1_W":
            pares_espectrales
            - pares_mismo_Z1_W,
        "pct_pares_espectrales_separados_Z1_W":
            (
                100
                * (
                    pares_espectrales
                    - pares_mismo_Z1_W
                )
                / pares_espectrales
                if pares_espectrales
                else np.nan
            ),
        "correlacion_pearson_Z1_W":
            correlacion_pearson_listas(
                zagreb,
                wiener
            ),
        "max_multiplicidad_cospectral":
            max(
                len(c)
                for c in clases_espectrales
            ),
    }

    representantes = []

    for k, clase_espectral in enumerate(
        clases_espectrales,
        start=1
    ):
        r = clase_espectral[0]

        representantes.append({
            "clase_entropia": numero,
            "clase_espectral_local": k,
            "multiplicidad_cospectral":
                len(clase_espectral),
            "indice_representante":
                r["indice"],
            "codigo_representante":
                r["codigo"],
            "entropia":
                r["entropia"],
            "zagreb1":
                r["zagreb1"],
            "wiener":
                r["wiener"],
        })

    colisiones = []

    for i in range(len(reps)):
        for j in range(i + 1, len(reps)):
            a = reps[i]
            b = reps[j]

            if (
                a["zagreb1"] == b["zagreb1"]
                and a["wiener"] == b["wiener"]
            ):
                colisiones.append({
                    "clase_entropia":
                        numero,
                    "entropia_representativa":
                        float(np.mean(entropias)),
                    "indice1":
                        a["indice"],
                    "indice2":
                        b["indice"],
                    "codigo1":
                        a["codigo"],
                    "codigo2":
                        b["codigo"],
                    "zagreb1":
                        a["zagreb1"],
                    "wiener":
                        a["wiener"],
                    "delta_entropia":
                        abs(
                            a["entropia"]
                            - b["entropia"]
                        ),
                })

    return fila, representantes, colisiones


def analizar_entropia_global_db(
    ruta,
    guardar_detalle=True,
    mostrar_progreso=True
):
    con = conectar_base(ruta)

    cursor = con.execute("""
        SELECT
            indice,
            codigo,
            entropia,
            zagreb1,
            wiener
        FROM arboles
        ORDER BY entropia, indice
    """)

    E = 0
    Q_total = 0

    n_S_Z1 = 0
    n_S_W = 0
    n_S_Z1_W = 0

    filas_entropia = []
    filas_ce = []
    representantes_ce = []
    colisiones_ce = []

    for clase in iterar_grupos_entropia_cursor(
        cursor,
        tol=EPS_ENTROPIA,
        etiqueta=(
            "Entropía"
            if mostrar_progreso
            else None
        )
    ):
        E += 1
        numero = E
        N_clase = len(clase)

        zagreb = [
            r["zagreb1"]
            for r in clase
        ]

        wiener = [
            r["wiener"]
            for r in clase
        ]

        n_S_Z1 += len(set(zagreb))
        n_S_W += len(set(wiener))
        n_S_Z1_W += len(
            set(zip(zagreb, wiener))
        )

        if N_clase == 1:
            clases_espectrales = [clase]
        else:
            clases_espectrales = (
                agrupar_por_espectro_optimizado(
                    clase,
                    tol=EPS_ESPECTRO
                )
            )

        Q = len(clases_espectrales)
        Q_total += Q

        if (
            guardar_detalle
            and N_clase >= 2
        ):
            entropias = [
                r["entropia"]
                for r in clase
            ]

            total_pares = (
                N_clase
                * (N_clase - 1)
                // 2
            )

            pares_cospectrales = (
                numero_pares_en_clases(
                    clases_espectrales
                )
            )

            sep_Z1, pct_Z1 = (
                pares_separados_por_campos(
                    clase,
                    "zagreb1"
                )
            )

            sep_W, pct_W = (
                pares_separados_por_campos(
                    clase,
                    "wiener"
                )
            )

            sep_ZW, pct_ZW = (
                pares_separados_por_campos(
                    clase,
                    ("zagreb1", "wiener")
                )
            )

            filas_entropia.append({
                "clase_entropia": numero,
                "n_arboles": N_clase,
                "entropia_representativa":
                    float(np.mean(entropias)),
                "entropia_min":
                    min(entropias),
                "entropia_max":
                    max(entropias),
                "rango_entropia":
                    max(entropias)
                    - min(entropias),
                "n_espectros_distintos": Q,
                "pares_totales":
                    total_pares,
                "pares_cospectrales":
                    pares_cospectrales,
                "pares_no_cospectrales":
                    total_pares
                    - pares_cospectrales,
                "zagreb_min":
                    min(zagreb),
                "zagreb_max":
                    max(zagreb),
                "rango_zagreb":
                    max(zagreb) - min(zagreb),
                "n_zagreb_distintos":
                    len(set(zagreb)),
                "wiener_min":
                    min(wiener),
                "wiener_max":
                    max(wiener),
                "rango_wiener":
                    max(wiener) - min(wiener),
                "n_wiener_distintos":
                    len(set(wiener)),
                "n_valores_Z1_W_distintos":
                    len(set(zip(zagreb, wiener))),
                "pares_separados_Z1":
                    sep_Z1,
                "porcentaje_pares_separados_Z1":
                    pct_Z1,
                "pares_separados_W":
                    sep_W,
                "porcentaje_pares_separados_W":
                    pct_W,
                "pares_separados_Z1_W":
                    sep_ZW,
                "porcentaje_pares_separados_Z1_W":
                    pct_ZW,
            })

        if Q >= 2:
            (
                fila_ce,
                reps,
                cols,
            ) = analizar_clase_ce(
                numero,
                clase,
                clases_espectrales
            )

            filas_ce.append(fila_ce)

            if guardar_detalle:
                representantes_ce.extend(reps)
                colisiones_ce.extend(cols)

    con.close()

    return {
        "E": E,
        "Q": Q_total,
        "n_S_Z1": n_S_Z1,
        "n_S_W": n_S_W,
        "n_S_Z1_W": n_S_Z1_W,
        "entropia_fija_df":
            pd.DataFrame(
                filas_entropia,
                columns=COLUMNAS_ENTROPIA_FIJA
            ),
        "ce_df":
            pd.DataFrame(
                filas_ce,
                columns=COLUMNAS_CE
            ),
        "representantes_ce_df":
            pd.DataFrame(
                representantes_ce,
                columns=COLUMNAS_REPRESENTANTES_CE
            ),
        "colisiones_ce_df":
            pd.DataFrame(
                colisiones_ce,
                columns=COLUMNAS_COLISIONES_CE
            ),
    }


def resumen_global_ce(clases_df):
    columnas = [
        "n_clases_coentropicas_no_cospectrales",
        "n_arboles_implicados",
        "n_espectros_implicados",
        "perdida_espectral_por_entropia",
        "n_clases_S_Z1",
        "n_clases_S_W",
        "n_clases_S_Z1_W",
        "D_Z1_espectral_global",
        "D_W_espectral_global",
        "D_Z1_W_espectral_global",
        "recuperacion_perdida_Z1",
        "recuperacion_perdida_W",
        "recuperacion_perdida_Z1_W",
        "pares_espectrales",
        "pct_pares_espectrales_separados_Z1",
        "pct_pares_espectrales_separados_W",
        "pct_pares_espectrales_separados_Z1_W",
    ]

    if clases_df.empty:
        return pd.DataFrame([{
            "n_clases_coentropicas_no_cospectrales": 0,
            "n_arboles_implicados": 0,
            "n_espectros_implicados": 0,
            "perdida_espectral_por_entropia": 0,
            "n_clases_S_Z1": 0,
            "n_clases_S_W": 0,
            "n_clases_S_Z1_W": 0,
            "D_Z1_espectral_global": np.nan,
            "D_W_espectral_global": np.nan,
            "D_Z1_W_espectral_global": np.nan,
            "recuperacion_perdida_Z1": np.nan,
            "recuperacion_perdida_W": np.nan,
            "recuperacion_perdida_Z1_W": np.nan,
            "pares_espectrales": 0,
            "pct_pares_espectrales_separados_Z1": np.nan,
            "pct_pares_espectrales_separados_W": np.nan,
            "pct_pares_espectrales_separados_Z1_W": np.nan,
        }], columns=columnas)

    C = len(clases_df)
    Q = int(
        clases_df[
            "n_espectros_distintos"
        ].sum()
    )
    N = int(
        clases_df[
            "n_arboles"
        ].sum()
    )

    clases_Z1 = int(
        clases_df[
            "n_zagreb_distintos"
        ].sum()
    )
    clases_W = int(
        clases_df[
            "n_wiener_distintos"
        ].sum()
    )
    clases_Z1_W = int(
        clases_df[
            "n_valores_Z1_W_distintos"
        ].sum()
    )

    perdida = Q - C

    pares = int(
        clases_df[
            "pares_espectrales"
        ].sum()
    )

    separados_Z1 = int(
        clases_df[
            "pares_espectrales_separados_Z1"
        ].sum()
    )
    separados_W = int(
        clases_df[
            "pares_espectrales_separados_W"
        ].sum()
    )
    separados_Z1_W = int(
        clases_df[
            "pares_espectrales_separados_Z1_W"
        ].sum()
    )

    return pd.DataFrame([{
        "n_clases_coentropicas_no_cospectrales":
            C,
        "n_arboles_implicados":
            N,
        "n_espectros_implicados":
            Q,
        "perdida_espectral_por_entropia":
            perdida,
        "n_clases_S_Z1":
            clases_Z1,
        "n_clases_S_W":
            clases_W,
        "n_clases_S_Z1_W":
            clases_Z1_W,
        "D_Z1_espectral_global":
            clases_Z1 / Q,
        "D_W_espectral_global":
            clases_W / Q,
        "D_Z1_W_espectral_global":
            clases_Z1_W / Q,
        "recuperacion_perdida_Z1":
            (
                (clases_Z1 - C)
                / perdida
                if perdida
                else np.nan
            ),
        "recuperacion_perdida_W":
            (
                (clases_W - C)
                / perdida
                if perdida
                else np.nan
            ),
        "recuperacion_perdida_Z1_W":
            (
                (clases_Z1_W - C)
                / perdida
                if perdida
                else np.nan
            ),
        "pares_espectrales":
            pares,
        "pct_pares_espectrales_separados_Z1":
            (
                100 * separados_Z1 / pares
                if pares
                else np.nan
            ),
        "pct_pares_espectrales_separados_W":
            (
                100 * separados_W / pares
                if pares
                else np.nan
            ),
        "pct_pares_espectrales_separados_Z1_W":
            (
                100
                * separados_Z1_W
                / pares
                if pares
                else np.nan
            ),
    }], columns=columnas)

## 6. Análisis con Zagreb o Wiener fijo

Para cada valor de $Z_1$ o $W$ se calculan la cantidad de árboles, las entropías y espectros distintos, los extremos y la dispersión de entropía, las coincidencias espectrales y los rangos del otro índice. Se cuantifican las pérdidas $N-Q$, $Q-E$ y $N-E$.


In [ ]:
def analizar_por_invariante_db(
    ruta,
    campo,
    campo_secundario,
    mostrar_progreso=True
):
    if campo not in {
        "zagreb1",
        "wiener"
    }:
        raise ValueError(
            "campo debe ser 'zagreb1' o 'wiener'"
        )

    if campo_secundario not in {
        "zagreb1",
        "wiener"
    }:
        raise ValueError(
            "campo_secundario inválido"
        )

    con = conectar_base(ruta)

    consulta = f"""
        SELECT
            indice,
            codigo,
            entropia,
            zagreb1,
            wiener
        FROM arboles
        ORDER BY {campo}, entropia, indice
    """

    cursor = con.execute(consulta)

    filas = []

    valor_actual = None
    grupo_entropia = []
    minimo_entropia_grupo = None

    N = 0
    E = 0
    Q = 0
    suma_S = 0.0
    suma_S2 = 0.0
    S_min = None
    S_max = None

    secundarios = set()

    pares_coentropicos = 0
    pares_cospectrales = 0

    procesados = 0
    siguiente_aviso = PROGRESO_CADA


    def reiniciar(valor):
        return {
            "valor": valor,
            "N": 0,
            "E": 0,
            "Q": 0,
            "suma_S": 0.0,
            "suma_S2": 0.0,
            "S_min": None,
            "S_max": None,
            "secundarios": set(),
            "pares_coentropicos": 0,
            "pares_cospectrales": 0,
        }


    def procesar_clase_entropy(
        clase,
        estado
    ):
        if not clase:
            return

        estado["E"] += 1
        m = len(clase)

        if m == 1:
            clases_espectrales = [clase]
        else:
            clases_espectrales = (
                agrupar_por_espectro_optimizado(
                    clase,
                    tol=EPS_ESPECTRO
                )
            )

        estado["Q"] += len(
            clases_espectrales
        )

        estado["pares_coentropicos"] += (
            m * (m - 1) // 2
        )

        estado["pares_cospectrales"] += (
            numero_pares_en_clases(
                clases_espectrales
            )
        )


    def guardar_estado(estado):
        if (
            estado is None
            or estado["N"] == 0
        ):
            return

        media = (
            estado["suma_S"]
            / estado["N"]
        )

        varianza = max(
            0.0,
            (
                estado["suma_S2"]
                / estado["N"]
            )
            - media**2
        )

        secundarios_lista = (
            estado["secundarios"]
        )

        filas.append({
            campo:
                estado["valor"],
            "n_arboles":
                estado["N"],
            "n_grupos_numericos_entropia":
                estado["E"],
            "n_espectros_distintos":
                estado["Q"],
            "poder_discriminante_entropia":
                estado["E"]
                / estado["N"],
            "perdida_arbol_a_espectro":
                estado["N"]
                - estado["Q"],
            "perdida_espectro_a_entropia":
                estado["Q"]
                - estado["E"],
            "perdida_total_arbol_a_entropia":
                estado["N"]
                - estado["E"],
            "entropia_min":
                estado["S_min"],
            "entropia_max":
                estado["S_max"],
            "entropia_media":
                media,
            "entropia_desviacion":
                math.sqrt(varianza),
            "rango_entropia":
                state_range(
                    estado["S_min"],
                    estado["S_max"]
                ),
            "pares_eps_coentropicos":
                estado[
                    "pares_coentropicos"
                ],
            "pares_cospectrales":
                estado[
                    "pares_cospectrales"
                ],
            "pares_eps_coentropicos_no_cospectrales":
                (
                    estado[
                        "pares_coentropicos"
                    ]
                    - estado[
                        "pares_cospectrales"
                    ]
                ),
            f"{campo_secundario}_min":
                min(secundarios_lista),
            f"{campo_secundario}_max":
                max(secundarios_lista),
            f"rango_{campo_secundario}":
                max(secundarios_lista)
                - min(secundarios_lista),
            f"n_{campo_secundario}_distintos":
                len(secundarios_lista),
        })


    def state_range(a, b):
        return (
            b - a
            if a is not None
            and b is not None
            else np.nan
        )


    estado = None

    for fila in cursor:
        r = registro_desde_fila(fila)
        valor = r[campo]

        if estado is None:
            estado = reiniciar(valor)
            grupo_entropia = []
            minimo_entropia_grupo = None

        if valor != estado["valor"]:
            procesar_clase_entropy(
                grupo_entropia,
                estado
            )
            guardar_estado(estado)

            estado = reiniciar(valor)
            grupo_entropia = []
            minimo_entropia_grupo = None

        S = r["entropia"]

        if not grupo_entropia:
            grupo_entropia = [r]
            minimo_entropia_grupo = S

        elif (
            S - minimo_entropia_grupo
            <= EPS_ENTROPIA
        ):
            grupo_entropia.append(r)

        else:
            procesar_clase_entropy(
                grupo_entropia,
                estado
            )
            grupo_entropia = [r]
            minimo_entropia_grupo = S

        estado["N"] += 1
        estado["suma_S"] += S
        estado["suma_S2"] += S * S

        estado["S_min"] = (
            S
            if estado["S_min"] is None
            else min(estado["S_min"], S)
        )

        estado["S_max"] = (
            S
            if estado["S_max"] is None
            else max(estado["S_max"], S)
        )

        estado["secundarios"].add(
            r[campo_secundario]
        )

        procesados += 1

        if (
            mostrar_progreso
            and PROGRESO_CADA > 0
            and procesados >= siguiente_aviso
        ):
            print(
                f"{campo} fijo: "
                f"{procesados:,} filas analizadas"
            )
            siguiente_aviso += PROGRESO_CADA

    if estado is not None:
        procesar_clase_entropy(
            grupo_entropia,
            estado
        )
        guardar_estado(estado)

    con.close()

    return pd.DataFrame(filas)

## 7. Relación Zagreb–Wiener

Se estudian las frecuencias conjuntas de $(Z_1,W)$, la complementariedad de ambos índices y su correlación lineal. Las estadísticas se obtienen mediante consultas agregadas en SQLite.


In [ ]:
def elegir2(c):
    return c * (c - 1) // 2


def conteos_marginales(
    con,
    campo
):
    return [
        (fila[0], int(fila[1]))
        for fila in con.execute(
            f"""
            SELECT {campo}, COUNT(*)
            FROM arboles
            GROUP BY {campo}
            ORDER BY {campo}
            """
        )
    ]


def rangos_promedio_por_fijo(
    con,
    fijo,
    variable
):
    rangos = []

    for fila in con.execute(
        f"""
        SELECT
            {fijo},
            MIN({variable}),
            MAX({variable})
        FROM arboles
        GROUP BY {fijo}
        """
    ):
        rangos.append(
            int(fila[2]) - int(fila[1])
        )

    if not rangos:
        return {
            "promedio": np.nan,
            "maximo": np.nan,
        }

    return {
        "promedio":
            float(np.mean(rangos)),
        "maximo":
            int(max(rangos)),
    }


def correlacion_pearson_sql(con):
    fila = con.execute("""
        SELECT
            COUNT(*) AS N,
            SUM(1.0 * zagreb1) AS sz,
            SUM(1.0 * wiener) AS sw,
            SUM(1.0 * zagreb1 * zagreb1) AS sz2,
            SUM(1.0 * wiener * wiener) AS sw2,
            SUM(1.0 * zagreb1 * wiener) AS szw
        FROM arboles
    """).fetchone()

    N = float(fila["N"])

    if N < 2:
        return np.nan

    sz = float(fila["sz"])
    sw = float(fila["sw"])
    sz2 = float(fila["sz2"])
    sw2 = float(fila["sw2"])
    szw = float(fila["szw"])

    cov_num = (
        szw - sz * sw / N
    )

    vz = (
        sz2 - sz * sz / N
    )

    vw = (
        sw2 - sw * sw / N
    )

    if vz <= 0 or vw <= 0:
        return np.nan

    return float(
        cov_num
        / math.sqrt(vz * vw)
    )


def resumen_relacion_zagreb_wiener_db(
    ruta,
    n
):
    con = conectar_base(ruta)

    N = int(
        con.execute(
            "SELECT COUNT(*) FROM arboles"
        ).fetchone()[0]
    )

    conteos_z = conteos_marginales(
        con,
        "zagreb1"
    )

    conteos_w = conteos_marginales(
        con,
        "wiener"
    )

    n_Z1 = len(conteos_z)
    n_W = len(conteos_w)

    pares_mismo_Z1 = sum(
        elegir2(c)
        for _, c in conteos_z
    )

    pares_mismo_W = sum(
        elegir2(c)
        for _, c in conteos_w
    )

    n_Z1_W = 0
    pares_mismo_Z1_W = 0

    for _, _, cantidad in con.execute("""
        SELECT
            zagreb1,
            wiener,
            COUNT(*)
        FROM arboles
        GROUP BY zagreb1, wiener
    """):
        c = int(cantidad)
        n_Z1_W += 1
        pares_mismo_Z1_W += elegir2(c)

    pearson = correlacion_pearson_sql(con)

    minimo_maximo = con.execute("""
        SELECT
            MIN(zagreb1),
            MAX(zagreb1),
            MIN(wiener),
            MAX(wiener)
        FROM arboles
    """).fetchone()

    rango_Z1 = (
        int(minimo_maximo[1])
        - int(minimo_maximo[0])
    )

    rango_W = (
        int(minimo_maximo[3])
        - int(minimo_maximo[2])
    )

    rango_W_dado_Z1 = (
        rangos_promedio_por_fijo(
            con,
            "zagreb1",
            "wiener"
        )
    )

    rango_Z1_dado_W = (
        rangos_promedio_por_fijo(
            con,
            "wiener",
            "zagreb1"
        )
    )

    con.close()

    resueltos_por_W = (
        pares_mismo_Z1
        - pares_mismo_Z1_W
    )

    resueltos_por_Z1 = (
        pares_mismo_W
        - pares_mismo_Z1_W
    )

    return {
        "n": n,
        "n_arboles": N,
        "correlacion_pearson_Z1_W":
            pearson,
        "n_valores_Z1":
            n_Z1,
        "n_valores_W":
            n_W,
        "n_valores_Z1_W_distintos":
            n_Z1_W,
        "pares_totales":
            elegir2(N),
        "pares_mismo_Z1":
            pares_mismo_Z1,
        "pares_mismo_W":
            pares_mismo_W,
        "pares_mismo_Z1_W":
            pares_mismo_Z1_W,
        "colisiones_Z1_resueltas_por_W":
            resueltos_por_W,
        "porcentaje_colisiones_Z1_resueltas_por_W":
            (
                100
                * resueltos_por_W
                / pares_mismo_Z1
                if pares_mismo_Z1
                else np.nan
            ),
        "colisiones_W_resueltas_por_Z1":
            resueltos_por_Z1,
        "porcentaje_colisiones_W_resueltas_por_Z1":
            (
                100
                * resueltos_por_Z1
                / pares_mismo_W
                if pares_mismo_W
                else np.nan
            ),
        "promedio_rango_W_dado_Z1":
            rango_W_dado_Z1[
                "promedio"
            ],
        "max_rango_W_dado_Z1":
            rango_W_dado_Z1[
                "maximo"
            ],
        "promedio_rango_Z1_dado_W":
            rango_Z1_dado_W[
                "promedio"
            ],
        "max_rango_Z1_dado_W":
            rango_Z1_dado_W[
                "maximo"
            ],
        "promedio_rango_relativo_W_dado_Z1":
            (
                rango_W_dado_Z1[
                    "promedio"
                ] / rango_W
                if rango_W
                else 0.0
            ),
        "max_rango_relativo_W_dado_Z1":
            (
                rango_W_dado_Z1[
                    "maximo"
                ] / rango_W
                if rango_W
                else 0.0
            ),
        "promedio_rango_relativo_Z1_dado_W":
            (
                rango_Z1_dado_W[
                    "promedio"
                ] / rango_Z1
                if rango_Z1
                else 0.0
            ),
        "max_rango_relativo_Z1_dado_W":
            (
                rango_Z1_dado_W[
                    "maximo"
                ] / rango_Z1
                if rango_Z1
                else 0.0
            ),
    }

## 8. Análisis del orden principal

Se calculan los resúmenes del orden seleccionado. Las gráficas de dispersión emplean una muestra de hasta `MAX_PUNTOS_GRAFICO` árboles; las estadísticas corresponden a la población completa.


In [ ]:
ruta_db_principal = generar_base_orden(
    NUM_VERTICES,
    mostrar_progreso=True
)

muestra_datos_df = obtener_muestra_base(
    ruta_db_principal
)

print(
    "Muestra para visualización:",
    f"{len(muestra_datos_df):,} filas"
)

display(muestra_datos_df.head())


print("\nAnálisis por entropía...")

analisis_entropia = (
    analizar_entropia_global_db(
        ruta_db_principal,
        guardar_detalle=True,
        mostrar_progreso=True
    )
)

entropia_fija_df = (
    analisis_entropia[
        "entropia_fija_df"
    ]
)

coentropicas_no_cospectrales_df = (
    analisis_entropia[
        "ce_df"
    ]
)

representantes_espectrales_ce_df = (
    analisis_entropia[
        "representantes_ce_df"
    ]
)

colisiones_Z1_W_ce_df = (
    analisis_entropia[
        "colisiones_ce_df"
    ]
)

resumen_coentropicas_no_cospectrales_df = (
    resumen_global_ce(
        coentropicas_no_cospectrales_df
    )
)


print("\nAnálisis con Zagreb fijo...")

resumen_zagreb_df = (
    analizar_por_invariante_db(
        ruta_db_principal,
        campo="zagreb1",
        campo_secundario="wiener"
    )
)


print("\nAnálisis con Wiener fijo...")

resumen_wiener_df = (
    analizar_por_invariante_db(
        ruta_db_principal,
        campo="wiener",
        campo_secundario="zagreb1"
    )
)


print("\nAnálisis Zagreb–Wiener...")

relacion_ZW = (
    resumen_relacion_zagreb_wiener_db(
        ruta_db_principal,
        NUM_VERTICES
    )
)

relacion_zagreb_wiener_df = (
    pd.DataFrame([relacion_ZW])
)

## 9. Poder discriminante y pérdida de información

Sean $N$ el número de árboles no isomorfos, $Q$ el número de espectros laplacianos distintos y $E$ el número de clases numéricas de entropía. Entonces

$$
N-E=(N-Q)+(Q-E).
$$

Se separa así la pérdida debida a cospectralidad de la pérdida adicional al sustituir el espectro por su entropía.


In [ ]:
N = int(
    relacion_ZW[
        "n_arboles"
    ]
)

E = int(
    analisis_entropia[
        "E"
    ]
)

Q = int(
    analisis_entropia[
        "Q"
    ]
)

n_Z1 = int(
    relacion_ZW[
        "n_valores_Z1"
    ]
)

n_W = int(
    relacion_ZW[
        "n_valores_W"
    ]
)

n_Z1_W = int(
    relacion_ZW[
        "n_valores_Z1_W_distintos"
    ]
)

n_S_Z1 = int(
    analisis_entropia[
        "n_S_Z1"
    ]
)

n_S_W = int(
    analisis_entropia[
        "n_S_W"
    ]
)

n_S_Z1_W = int(
    analisis_entropia[
        "n_S_Z1_W"
    ]
)


descriptores = [
    ("Entropía S", E),
    ("Z1", n_Z1),
    ("Wiener W", n_W),
    ("(Z1, W)", n_Z1_W),
    ("(S, Z1)", n_S_Z1),
    ("(S, W)", n_S_W),
    ("(S, Z1, W)", n_S_Z1_W),
]


resumen_discriminacion_df = pd.DataFrame([
    {
        "descriptor": nombre,
        "n_clases": clases,
        "n_arboles": N,
        "poder_discriminante":
            clases / N,
        "porcentaje_discriminante":
            100 * clases / N,
    }
    for nombre, clases in descriptores
])


resumen_perdida_global_df = pd.DataFrame([{
    "n": NUM_VERTICES,
    "n_arboles": N,
    "n_espectros_distintos": Q,
    "n_grupos_numericos_entropia": E,
    "perdida_arbol_a_espectro":
        N - Q,
    "perdida_espectro_a_entropia":
        Q - E,
    "perdida_total_arbol_a_entropia":
        N - E,
    "retencion_espectral":
        Q / N,
    "retencion_entropia_desde_arbol":
        E / N,
    "retencion_entropia_desde_espectro":
        E / Q
        if Q else np.nan,
}])


print("Poder discriminante global:")
display(resumen_discriminacion_df)

print("Descomposición global de la pérdida:")
display(resumen_perdida_global_df)

print("Resumen con Z1 fijo:")
display(resumen_zagreb_df.head(50))

print("Resumen con Wiener fijo:")
display(resumen_wiener_df.head(50))

print("Clases de entropía repetida:")
display(entropia_fija_df.head(50))

print(
    "Clases coentrópicas con espectros distintos:"
)
display(
    coentropicas_no_cospectrales_df.head(50)
)

print(
    "Colisiones entre espectros distintos "
    "que conservan simultáneamente (Z1,W):"
)
display(colisiones_Z1_W_ce_df.head(50))

## 10. Evolución con el orden $n$

Se calculan los indicadores por orden y se presenta su evolución. Las bases SQLite existentes se reutilizan; las ausentes se generan y almacenan por lotes.


In [ ]:
def resumen_orden_db(
    n,
    ruta,
    analisis_entropia_previo=None,
    relacion_previa=None
):
    if analisis_entropia_previo is None:
        analisis_S = (
            analizar_entropia_global_db(
                ruta,
                guardar_detalle=False,
                mostrar_progreso=False
            )
        )
    else:
        analisis_S = (
            analisis_entropia_previo
        )

    if relacion_previa is None:
        relacion = (
            resumen_relacion_zagreb_wiener_db(
                ruta,
                n
            )
        )
    else:
        relacion = relacion_previa

    N = int(
        relacion["n_arboles"]
    )

    E = int(
        analisis_S["E"]
    )

    Q = int(
        analisis_S["Q"]
    )

    ce_resumen = resumen_global_ce(
        analisis_S["ce_df"]
    ).iloc[0]

    fila = {
        "n": n,
        "n_arboles": N,

        "n_clases_S":
            E,

        "n_espectros_distintos":
            Q,

        "n_clases_Z1":
            relacion["n_valores_Z1"],

        "n_clases_W":
            relacion["n_valores_W"],

        "n_clases_Z1_W":
            relacion[
                "n_valores_Z1_W_distintos"
            ],

        "D_S":
            E / N,

        "D_Z1":
            relacion["n_valores_Z1"]
            / N,

        "D_W":
            relacion["n_valores_W"]
            / N,

        "D_Z1_W":
            relacion[
                "n_valores_Z1_W_distintos"
            ] / N,

        "D_S_Z1":
            analisis_S["n_S_Z1"]
            / N,

        "D_S_W":
            analisis_S["n_S_W"]
            / N,

        "D_S_Z1_W":
            analisis_S["n_S_Z1_W"]
            / N,

        "correlacion_pearson_Z1_W":
            relacion[
                "correlacion_pearson_Z1_W"
            ],

        "pares_mismo_Z1":
            relacion[
                "pares_mismo_Z1"
            ],

        "pares_mismo_W":
            relacion[
                "pares_mismo_W"
            ],

        "pares_mismo_Z1_W":
            relacion[
                "pares_mismo_Z1_W"
            ],

        "pct_colisiones_Z1_resueltas_por_W":
            relacion[
                "porcentaje_colisiones_Z1_resueltas_por_W"
            ],

        "pct_colisiones_W_resueltas_por_Z1":
            relacion[
                "porcentaje_colisiones_W_resueltas_por_Z1"
            ],

        "promedio_rango_W_dado_Z1":
            relacion[
                "promedio_rango_W_dado_Z1"
            ],

        "max_rango_W_dado_Z1":
            relacion[
                "max_rango_W_dado_Z1"
            ],

        "promedio_rango_Z1_dado_W":
            relacion[
                "promedio_rango_Z1_dado_W"
            ],

        "max_rango_Z1_dado_W":
            relacion[
                "max_rango_Z1_dado_W"
            ],

        "promedio_rango_relativo_W_dado_Z1":
            relacion[
                "promedio_rango_relativo_W_dado_Z1"
            ],

        "max_rango_relativo_W_dado_Z1":
            relacion[
                "max_rango_relativo_W_dado_Z1"
            ],

        "promedio_rango_relativo_Z1_dado_W":
            relacion[
                "promedio_rango_relativo_Z1_dado_W"
            ],

        "max_rango_relativo_Z1_dado_W":
            relacion[
                "max_rango_relativo_Z1_dado_W"
            ],
    }

    prefijos = {
        "n_clases_coentropicas_no_cospectrales":
            "CE_n_clases",
        "n_arboles_implicados":
            "CE_n_arboles_implicados",
        "n_espectros_implicados":
            "CE_n_espectros_implicados",
        "perdida_espectral_por_entropia":
            "CE_perdida_espectral_entropia",
        "D_Z1_espectral_global":
            "CE_D_Z1_espectral",
        "D_W_espectral_global":
            "CE_D_W_espectral",
        "D_Z1_W_espectral_global":
            "CE_D_Z1_W_espectral",
        "recuperacion_perdida_Z1":
            "CE_recuperacion_perdida_Z1",
        "recuperacion_perdida_W":
            "CE_recuperacion_perdida_W",
        "recuperacion_perdida_Z1_W":
            "CE_recuperacion_perdida_Z1_W",
        "pct_pares_espectrales_separados_Z1":
            "CE_pct_pares_separados_Z1",
        "pct_pares_espectrales_separados_W":
            "CE_pct_pares_separados_W",
        "pct_pares_espectrales_separados_Z1_W":
            "CE_pct_pares_separados_Z1_W",
    }

    for origen, destino in prefijos.items():
        fila[destino] = (
            ce_resumen[origen]
        )

    return fila


def estudiar_tendencia_ordenes(
    orden_min,
    orden_max
):
    filas = []

    for n in range(
        orden_min,
        orden_max + 1
    ):
        print(
            f"\nTendencia: preparando n={n}"
        )

        ruta = generar_base_orden(
            n,
            mostrar_progreso=True
        )

        if n == NUM_VERTICES:
            fila = resumen_orden_db(
                n,
                ruta,
                analisis_entropia_previo=
                    analisis_entropia,
                relacion_previa=
                    relacion_ZW
            )
        else:
            fila = resumen_orden_db(
                n,
                ruta
            )

        filas.append(fila)

        gc.collect()

    return pd.DataFrame(filas)


if CALCULAR_TENDENCIA:
    tendencia_df = (
        estudiar_tendencia_ordenes(
            ORDEN_MIN_TENDENCIA,
            ORDEN_MAX_TENDENCIA
        )
    )
else:
    tendencia_df = pd.DataFrame()


columnas_tendencia_ce = [
    "n",
    "CE_n_clases",
    "CE_n_arboles_implicados",
    "CE_n_espectros_implicados",
    "CE_perdida_espectral_entropia",
    "CE_D_Z1_espectral",
    "CE_D_W_espectral",
    "CE_D_Z1_W_espectral",
    "CE_recuperacion_perdida_Z1",
    "CE_recuperacion_perdida_W",
    "CE_recuperacion_perdida_Z1_W",
    "CE_pct_pares_separados_Z1",
    "CE_pct_pares_separados_W",
    "CE_pct_pares_separados_Z1_W",
]


if not tendencia_df.empty:
    tendencia_coentropicas_no_cospectrales_df = (
        tendencia_df[
            columnas_tendencia_ce
        ].copy()
    )
else:
    tendencia_coentropicas_no_cospectrales_df = (
        pd.DataFrame(
            columns=columnas_tendencia_ce
        )
    )


display(tendencia_df)

print(
    "Tendencia de clases "
    "coentrópicas con espectros distintos:"
)

display(
    tendencia_coentropicas_no_cospectrales_df
)

## 11. Visualización

Las distribuciones se representan mediante muestras de árboles. Las curvas de resúmenes y tendencias utilizan estadísticas de la enumeración completa.


In [ ]:
def figura_entropia_vs_invariante(
    muestra_df,
    campo,
    etiqueta,
    titulo
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.scatter(
        muestra_df[campo],
        muestra_df["entropia"],
        s=9,
        alpha=0.35
    )

    ax.set_xlabel(etiqueta)
    ax.set_ylabel(
        "Entropía de von Neumann"
    )
    ax.set_title(titulo)
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_wiener_vs_zagreb(
    muestra_df
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.scatter(
        muestra_df["wiener"],
        muestra_df["zagreb1"],
        s=9,
        alpha=0.35
    )

    ax.set_xlabel(
        r"Índice de Wiener $W$"
    )

    ax.set_ylabel(
        r"Primer índice de Zagreb $Z_1$"
    )

    ax.set_title(
        f"Relación Wiener–Zagreb, n={NUM_VERTICES}"
    )

    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_rango_entropia(
    resumen_df,
    campo,
    etiqueta,
    titulo
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        resumen_df[campo],
        resumen_df["entropia_min"],
        label="Entropía mínima"
    )

    ax.plot(
        resumen_df[campo],
        resumen_df["entropia_max"],
        label="Entropía máxima"
    )

    ax.fill_between(
        resumen_df[campo],
        resumen_df["entropia_min"],
        resumen_df["entropia_max"],
        alpha=0.2
    )

    ax.set_xlabel(etiqueta)
    ax.set_ylabel(
        "Entropía de von Neumann"
    )
    ax.set_title(titulo)
    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_rango_secundario(
    resumen_df,
    campo,
    campo_secundario,
    etiqueta_x,
    etiqueta_y,
    titulo
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    minimo = resumen_df[
        f"{campo_secundario}_min"
    ]

    maximo = resumen_df[
        f"{campo_secundario}_max"
    ]

    ax.plot(
        resumen_df[campo],
        minimo,
        label="Mínimo"
    )

    ax.plot(
        resumen_df[campo],
        maximo,
        label="Máximo"
    )

    ax.fill_between(
        resumen_df[campo],
        minimo,
        maximo,
        alpha=0.2
    )

    ax.set_xlabel(etiqueta_x)
    ax.set_ylabel(etiqueta_y)
    ax.set_title(titulo)
    ax.legend()
    ax.grid(alpha=0.25)

    fig.tight_layout()

    return fig


def figura_discriminacion_global(
    resumen_df
):
    datos = resumen_df.sort_values(
        "porcentaje_discriminante"
    )

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.barh(
        datos["descriptor"],
        datos["porcentaje_discriminante"]
    )

    ax.set_xlabel(
        "Poder discriminante (%)"
    )

    ax.set_title(
        f"Poder discriminante, n={NUM_VERTICES}"
    )

    fig.tight_layout()

    return fig


def figura_perdidas_por_zagreb(
    resumen_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    x = resumen_df["zagreb1"]

    ax.plot(
        x,
        resumen_df[
            "perdida_arbol_a_espectro"
        ],
        label=r"$N-Q$: árbol a espectro"
    )

    ax.plot(
        x,
        resumen_df[
            "perdida_espectro_a_entropia"
        ],
        label=r"$Q-E$: espectro a entropía"
    )

    ax.plot(
        x,
        resumen_df[
            "perdida_total_arbol_a_entropia"
        ],
        label=r"$N-E$: pérdida total"
    )

    ax.set_xlabel(
        r"Primer índice de Zagreb $Z_1$"
    )

    ax.set_ylabel(
        "Cantidad de distinciones perdidas"
    )

    ax.set_title(
        "Descomposición de la pérdida dentro de cada clase de Zagreb"
    )

    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_tendencia_discriminacion(
    df
):
    fig, ax = plt.subplots(
        figsize=(11, 6.5)
    )

    columnas = [
        ("D_S", "S"),
        ("D_Z1", "Z1"),
        ("D_W", "W"),
        ("D_Z1_W", "(Z1, W)"),
        ("D_S_Z1", "(S, Z1)"),
        ("D_S_W", "(S, W)"),
        ("D_S_Z1_W", "(S, Z1, W)"),
    ]

    for columna, etiqueta in columnas:
        ax.plot(
            df["n"],
            100 * df[columna],
            marker="o",
            label=etiqueta
        )

    ax.set_xlabel(
        "Número de vértices n"
    )
    ax.set_ylabel(
        "Poder discriminante (%)"
    )
    ax.set_title(
        "Evolución del poder discriminante"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_tendencia_resolucion_colisiones(
    df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        df["n"],
        df[
            "pct_colisiones_Z1_resueltas_por_W"
        ],
        marker="o",
        label="W resuelve colisiones de Z1"
    )

    ax.plot(
        df["n"],
        df[
            "pct_colisiones_W_resueltas_por_Z1"
        ],
        marker="s",
        label="Z1 resuelve colisiones de W"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )
    ax.set_ylabel(
        "Colisiones resueltas (%)"
    )
    ax.set_title(
        "Complementariedad entre Zagreb y Wiener"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_wiener_vs_zagreb_ce(
    representantes_df
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    if representantes_df.empty:
        ax.text(
            0.5,
            0.5,
            "No hay clases coentrópicas con espectros distintos",
            ha="center",
            va="center",
            transform=ax.transAxes
        )
    else:
        ax.scatter(
            representantes_df["wiener"],
            representantes_df["zagreb1"],
            s=18,
            alpha=0.55
        )

    ax.set_xlabel(
        r"Índice de Wiener $W$"
    )
    ax.set_ylabel(
        r"Primer índice de Zagreb $Z_1$"
    )
    ax.set_title(
        "Wiener–Zagreb en espectros distintos "
        "dentro de clases coentrópicas"
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_discriminacion_espectral_ce(
    clases_df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    if clases_df.empty:
        ax.text(
            0.5,
            0.5,
            "No hay clases coentrópicas con espectros distintos",
            ha="center",
            va="center",
            transform=ax.transAxes
        )
    else:
        datos = clases_df.sort_values(
            "entropia_representativa"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_Z1_espectral"],
            marker="o",
            label="Z1"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_W_espectral"],
            marker="s",
            label="W"
        )

        ax.plot(
            datos["entropia_representativa"],
            100 * datos["D_Z1_W_espectral"],
            marker="^",
            label="(Z1, W)"
        )

        ax.legend()

    ax.set_xlabel(
        "Entropía representativa de la clase"
    )
    ax.set_ylabel(
        "Poder discriminante espectral (%)"
    )
    ax.set_title(
        "Discriminación de espectros dentro de clases coentrópicas"
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_recuperacion_global_ce(
    resumen_df
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    fila = resumen_df.iloc[0]

    etiquetas = [
        "Z1",
        "W",
        "(Z1, W)",
    ]

    valores = [
        100 * fila[
            "recuperacion_perdida_Z1"
        ],
        100 * fila[
            "recuperacion_perdida_W"
        ],
        100 * fila[
            "recuperacion_perdida_Z1_W"
        ],
    ]

    ax.bar(
        etiquetas,
        valores
    )

    ax.set_ylabel(
        "Pérdida espectral recuperada (%)"
    )
    ax.set_title(
        "Información espectral recuperada "
        "dentro de las colisiones de entropía"
    )
    ax.set_ylim(0, 105)
    fig.tight_layout()

    return fig


def figura_tendencia_recuperacion_ce(
    df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        df["n"],
        100 * df[
            "CE_recuperacion_perdida_Z1"
        ],
        marker="o",
        label="Z1"
    )

    ax.plot(
        df["n"],
        100 * df[
            "CE_recuperacion_perdida_W"
        ],
        marker="s",
        label="W"
    )

    ax.plot(
        df["n"],
        100 * df[
            "CE_recuperacion_perdida_Z1_W"
        ],
        marker="^",
        label="(Z1, W)"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )
    ax.set_ylabel(
        "Pérdida espectral recuperada (%)"
    )
    ax.set_title(
        "Evolución de la recuperación dentro "
        "de clases coentrópicas no cospectrales"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_tendencia_clases_ce(
    df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        df["n"],
        df["CE_n_clases"],
        marker="o",
        label="Clases de entropía"
    )

    ax.plot(
        df["n"],
        df["CE_n_espectros_implicados"],
        marker="s",
        label="Espectros implicados"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )
    ax.set_ylabel(
        "Cantidad"
    )
    ax.set_title(
        "Evolución de las colisiones espectrales de entropía"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig


def figura_tendencia_separacion_pares_ce(
    df
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        df["n"],
        df[
            "CE_pct_pares_separados_Z1"
        ],
        marker="o",
        label="Z1"
    )

    ax.plot(
        df["n"],
        df[
            "CE_pct_pares_separados_W"
        ],
        marker="s",
        label="W"
    )

    ax.plot(
        df["n"],
        df[
            "CE_pct_pares_separados_Z1_W"
        ],
        marker="^",
        label="(Z1, W)"
    )

    ax.set_xlabel(
        "Número de vértices n"
    )
    ax.set_ylabel(
        "Pares espectrales separados (%)"
    )
    ax.set_title(
        "Separación de espectros dentro de clases coentrópicas"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    return fig

## 12. Exportación de resultados

Los registros individuales permanecen en SQLite. Se exportan tablas de análisis en CSV, resúmenes compatibles con Excel y figuras en PNG. La exportación masiva a CSV es opcional.


In [ ]:
# Dependencias de exportación
import os
import csv


def exportar_sqlite_a_csv(
    ruta_db,
    ruta_csv
):
    con = conectar_base(ruta_db)

    cursor = con.execute("""
        SELECT
            indice,
            codigo,
            entropia,
            zagreb1,
            wiener
        FROM arboles
        ORDER BY indice
    """)

    with open(
        ruta_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as archivo:
        escritor = csv.writer(archivo)

        escritor.writerow([
            "indice",
            "codigo",
            "entropia",
            "zagreb1",
            "wiener",
        ])

        for fila in cursor:
            escritor.writerow([
                fila["indice"],
                fila["codigo"],
                fila["entropia"],
                fila["zagreb1"],
                fila["wiener"],
            ])

    con.close()


tablas = {
    "Muestra_datos":
        muestra_datos_df,

    "Discriminacion":
        resumen_discriminacion_df,

    "Perdida_global":
        resumen_perdida_global_df,

    "Zagreb_fijo":
        resumen_zagreb_df,

    "Wiener_fijo":
        resumen_wiener_df,

    "Entropia_fija":
        entropia_fija_df,

    "Relacion_Zagreb_Wiener":
        relacion_zagreb_wiener_df,

    "CE_no_cospectrales":
        coentropicas_no_cospectrales_df,

    "CE_representantes_espectrales":
        representantes_espectrales_ce_df,

    "CE_colisiones_Z1_W":
        colisiones_Z1_W_ce_df,

    "CE_resumen_global":
        resumen_coentropicas_no_cospectrales_df,

    "Tendencia_ordenes":
        tendencia_df,

    "Tendencia_CE":
        tendencia_coentropicas_no_cospectrales_df,
}


# Tablas CSV

for nombre, df in tablas.items():
    ruta_csv = os.path.join(
        CARPETA_TABLAS,
        f"{nombre}.csv"
    )

    df.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig"
    )


# Resúmenes en Excel

ruta_excel = os.path.join(
    CARPETA_TABLAS,
    f"analisis_resumido_orden_{NUM_VERTICES}.xlsx"
)


with pd.ExcelWriter(
    ruta_excel,
    engine="openpyxl"
) as writer:

    for nombre, df in tablas.items():
        # Reserva de fila para encabezados.
        if len(df) + 1 > MAX_FILAS_EXCEL:
            print(
                f"Excel: se omite {nombre} "
                f"({len(df):,} filas). "
                "El CSV sí fue guardado."
            )
            continue

        df.to_excel(
            writer,
            sheet_name=nombre[:31],
            index=False
        )


# Exportación CSV de registros individuales

ruta_csv_masivo = os.path.join(
    CARPETA_DATOS,
    f"arboles_orden_{NUM_VERTICES}.csv"
)


if EXPORTAR_CSV_MASIVO:
    print(
        "Exportación de CSV individual. "
        ""
    )

    exportar_sqlite_a_csv(
        ruta_db_principal,
        ruta_csv_masivo
    )


# Exportación de figuras

def guardar_y_mostrar_figura(
    fig,
    nombre_archivo
):
    ruta_figura = os.path.join(
        CARPETA_IMAGENES,
        nombre_archivo
    )

    fig.savefig(
        ruta_figura,
        dpi=300,
        bbox_inches="tight"
    )

    display(fig)
    plt.close(fig)


guardar_y_mostrar_figura(
    figura_entropia_vs_invariante(
        muestra_datos_df,
        "zagreb1",
        r"Primer índice de Zagreb $Z_1$",
        f"Entropía vs. Zagreb, n={NUM_VERTICES}"
    ),
    "01_entropia_vs_zagreb.png"
)


guardar_y_mostrar_figura(
    figura_entropia_vs_invariante(
        muestra_datos_df,
        "wiener",
        r"Índice de Wiener $W$",
        f"Entropía vs. Wiener, n={NUM_VERTICES}"
    ),
    "02_entropia_vs_wiener.png"
)


guardar_y_mostrar_figura(
    figura_wiener_vs_zagreb(
        muestra_datos_df
    ),
    "03_wiener_vs_zagreb.png"
)


guardar_y_mostrar_figura(
    figura_rango_entropia(
        resumen_zagreb_df,
        "zagreb1",
        r"Primer índice de Zagreb $Z_1$",
        "Rango de entropía con Zagreb fijo"
    ),
    "04_rango_entropia_Z1_fijo.png"
)


guardar_y_mostrar_figura(
    figura_rango_entropia(
        resumen_wiener_df,
        "wiener",
        r"Índice de Wiener $W$",
        "Rango de entropía con Wiener fijo"
    ),
    "05_rango_entropia_W_fijo.png"
)


guardar_y_mostrar_figura(
    figura_rango_secundario(
        resumen_zagreb_df,
        "zagreb1",
        "wiener",
        r"Primer índice de Zagreb $Z_1$",
        r"Índice de Wiener $W$",
        "Variación de Wiener con Zagreb fijo"
    ),
    "06_rango_W_Z1_fijo.png"
)


guardar_y_mostrar_figura(
    figura_rango_secundario(
        resumen_wiener_df,
        "wiener",
        "zagreb1",
        r"Índice de Wiener $W$",
        r"Primer índice de Zagreb $Z_1$",
        "Variación de Zagreb con Wiener fijo"
    ),
    "07_rango_Z1_W_fijo.png"
)


guardar_y_mostrar_figura(
    figura_discriminacion_global(
        resumen_discriminacion_df
    ),
    "08_poder_discriminante.png"
)


guardar_y_mostrar_figura(
    figura_perdidas_por_zagreb(
        resumen_zagreb_df
    ),
    "09_perdidas_por_zagreb.png"
)


if not tendencia_df.empty:
    guardar_y_mostrar_figura(
        figura_tendencia_discriminacion(
            tendencia_df
        ),
        "10_tendencia_discriminacion.png"
    )


    guardar_y_mostrar_figura(
        figura_tendencia_resolucion_colisiones(
            tendencia_df
        ),
        "11_tendencia_resolucion_colisiones_ZW.png"
    )


guardar_y_mostrar_figura(
    figura_wiener_vs_zagreb_ce(
        representantes_espectrales_ce_df
    ),
    "12_wiener_vs_zagreb_CE_no_cospectrales.png"
)


guardar_y_mostrar_figura(
    figura_discriminacion_espectral_ce(
        coentropicas_no_cospectrales_df
    ),
    "13_discriminacion_espectral_CE.png"
)


guardar_y_mostrar_figura(
    figura_recuperacion_global_ce(
        resumen_coentropicas_no_cospectrales_df
    ),
    "14_recuperacion_global_CE.png"
)


if not tendencia_coentropicas_no_cospectrales_df.empty:
    guardar_y_mostrar_figura(
        figura_tendencia_recuperacion_ce(
            tendencia_coentropicas_no_cospectrales_df
        ),
        "15_tendencia_recuperacion_CE.png"
    )

    guardar_y_mostrar_figura(
        figura_tendencia_clases_ce(
            tendencia_coentropicas_no_cospectrales_df
        ),
        "16_tendencia_clases_CE.png"
    )

    guardar_y_mostrar_figura(
        figura_tendencia_separacion_pares_ce(
            tendencia_coentropicas_no_cospectrales_df
        ),
        "17_tendencia_separacion_pares_CE.png"
    )


print("\nExportación terminada.")
print(
    "Base SQLite completa:",
    os.path.abspath(
        ruta_db_principal
    )
)
print(
    "Tablas de resumen:",
    os.path.abspath(
        CARPETA_TABLAS
    )
)
print(
    "Imágenes:",
    os.path.abspath(
        CARPETA_IMAGENES
    )
)
print(
    "Excel de resúmenes:",
    os.path.abspath(
        ruta_excel
    )
)

if EXPORTAR_CSV_MASIVO:
    print(
        "CSV masivo:",
        os.path.abspath(
            ruta_csv_masivo
        )
    )